In [ ]:
"""
Rice Leaf Disease Classification - Model SE
============================================
Architecture: Residual Blocks + Dense Blocks + SE Blocks

Hybrid CNN combining:
    - Residual connections (ResNet style)
    - Dense connectivity (DenseNet style)
    - Squeeze-and-Excitation attention (channel attention)

Train on Google Colab.
"""

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix
)

# ============================================================
# REPRODUCIBILITY
# ============================================================
import random
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================
# CONFIGURATION
# ============================================================
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 120

# Colab paths
TRAIN_DIR = "/content/drive/MyDrive/rice_leaf_split/train"
VAL_DIR   = "/content/drive/MyDrive/rice_leaf_split/validation"
TEST_DIR  = "/content/drive/MyDrive/rice_leaf_split/test"

MODEL_SAVE_PATH = "/content/drive/MyDrive/model_SE_best.h5"
HISTORY_PATH = "/content/drive/MyDrive/history_SE.json"

# ============================================================
# DATA GENERATORS
# ============================================================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.10,
    width_shift_range=0.08,
    height_shift_range=0.08,
    horizontal_flip=True,
    brightness_range=[0.88, 1.12],
    fill_mode='reflect'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False
)

class_names = list(train_generator.class_indices.keys())
NUM_CLASSES = len(class_names)

print(f"Classes: {NUM_CLASSES}")
print(f"Train: {train_generator.samples}")
print(f"Val:   {val_generator.samples}")
print(f"Test:  {test_generator.samples}")

# ============================================================
# MODEL BLOCKS
# ============================================================
def residual_block(x, filters):
    """Residual block (ResNet style)."""
    shortcut = Conv2D(filters, 1, padding="same")(x)
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(0.1)(x)
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Add()([x, shortcut])
    x = LeakyReLU(0.1)(x)
    return x

def dense_block(x, growth_rate=32):
    """Dense block (DenseNet style)."""
    y = Conv2D(growth_rate, 3, padding="same", activation="relu")(x)
    x = Concatenate()([x, y])
    y = Conv2D(growth_rate, 3, padding="same", activation="relu")(x)
    x = Concatenate()([x, y])
    return x

def se_block(x):
    """Squeeze-and-Excitation block (channel attention)."""
    filters = x.shape[-1]
    se = GlobalAveragePooling2D()(x)
    se = Dense(filters // 4, activation="relu")(se)
    se = Dense(filters, activation="sigmoid")(se)
    se = Reshape((1, 1, filters))(se)
    return Multiply()([x, se])

# ============================================================
# BUILD MODEL
# ============================================================
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = Conv2D(64, 3, padding="same")(inputs)
x = BatchNormalization()(x)
x = LeakyReLU(0.1)(x)

# Block 1: Residual
x = residual_block(x, 64)
x = MaxPooling2D()(x)

# Block 2: Dense + SE
x = dense_block(x, growth_rate=32)
x = se_block(x)
x = MaxPooling2D()(x)

# Block 3: Residual
x = residual_block(x, 128)
x = MaxPooling2D()(x)

# Block 4: Dense + SE
x = dense_block(x, growth_rate=32)
x = se_block(x)
x = MaxPooling2D()(x)

# Block 5: Residual (no pool)
x = residual_block(x, 256)

# Classification Head
x = GlobalAveragePooling2D()(x)
x = Dense(512)(x)
x = BatchNormalization()(x)
x = LeakyReLU(0.1)(x)
x = Dropout(0.5)(x)

x = Dense(256)(x)
x = LeakyReLU(0.1)(x)
x = Dropout(0.3)(x)

outputs = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs, outputs, name="Hybrid_SE")

# Compile
optimizer = AdamW(learning_rate=1e-4, weight_decay=1e-4)
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# CLASS WEIGHTS
# ============================================================
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights))

# ============================================================
# CALLBACKS
# ============================================================
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.3,
                      min_lr=1e-7, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_loss',
                    save_best_only=True, verbose=1)
]

# ============================================================
# TRAIN
# ============================================================
print("=" * 70)
print("TRAINING: Hybrid (Residual + Dense + SE)")
print("=" * 70)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)

# Save history
history_dict = {
    'accuracy': [float(x) for x in history.history['accuracy']],
    'val_accuracy': [float(x) for x in history.history['val_accuracy']],
    'loss': [float(x) for x in history.history['loss']],
    'val_loss': [float(x) for x in history.history['val_loss']],
}
with open(HISTORY_PATH, 'w') as f:
    json.dump(history_dict, f)

print(f"\n✅ Model saved: {MODEL_SAVE_PATH}")
print(f"✅ History saved: {HISTORY_PATH}")

# ============================================================
# EVALUATE ON TEST SET
# ============================================================
print("\n" + "=" * 70)
print("TEST SET EVALUATION")
print("=" * 70)

test_pred = model.predict(test_generator, verbose=1)
test_pred_classes = np.argmax(test_pred, axis=1)
test_true = test_generator.classes

test_acc = accuracy_score(test_true, test_pred_classes) * 100
print(f"\nTest Accuracy: {test_acc:.2f}%")
print("\nClassification Report:")
print(classification_report(test_true, test_pred_classes,
                            target_names=class_names))

# Confusion matrix
cm = confusion_matrix(test_true, test_pred_classes)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Model SE (Test Set)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/cm_SE.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved: cm_SE.png")

In [ ]:
ARCHITECTURE OVERVIEW
Input (224×224×3)
    ↓
Stem: Conv2D(64) + BN + LeakyReLU
    ↓
Block 1: Residual Block (64 filters) → MaxPool
    ↓
Block 2: Dense Block (growth=32) → SE Block → MaxPool
    ↓
Block 3: Residual Block (128 filters) → MaxPool
    ↓
Block 4: Dense Block (growth=32) → SE Block → MaxPool
    ↓
Block 5: Residual Block (256 filters)
    ↓
GlobalAveragePooling2D
    ↓
Dense(512) + BN + LeakyReLU + Dropout(0.5)
    ↓
Dense(256) + LeakyReLU + Dropout(0.3)
    ↓
Dense(12) + Softmax

In [ ]:
"""
================================================================================
A Hybrid Deep Learning Architecture Combining Residual, Dense, and Attention
Mechanisms for Rice Leaf Disease Classification
================================================================================

Architecture: Hybrid CNN
    - Residual Blocks (ResNet-style)
    - Dense Blocks (DenseNet-style)
    - Squeeze-and-Excitation Blocks (Attention)

Dataset: Rice Leaf Disease (12 classes)
    - Train:      3,656 images (after duplicate removal)
    - Validation:   455 images
    - Test:         464 images

Environment: Google Colab (GPU: NVIDIA Tesla T4)

Author: [Your Name]
Date:   2026
================================================================================
"""

# ==============================================================================
# IMPORTS
# ==============================================================================
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, LeakyReLU, ReLU, Add, Concatenate,
    MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, Reshape, Multiply
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import (
    ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix
)

# ==============================================================================
# REPRODUCIBILITY — Fix all random seeds
# ==============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ==============================================================================
# CONFIGURATION
# ==============================================================================
IMG_SIZE = 224
IMG_CHANNELS = 3           # RGB
BATCH_SIZE = 32
EPOCHS = 120
NUM_CLASSES = 12

# Paths (Colab)
BASE_DIR = "/content/drive/MyDrive/rice_leaf_split"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "validation")
TEST_DIR  = os.path.join(BASE_DIR, "test")

# Save paths
MODEL_SAVE_PATH  = "/content/drive/MyDrive/hybrid_rds_best.h5"
HISTORY_SAVE_PATH = "/content/drive/MyDrive/history_hybrid_rds.json"
CM_SAVE_PATH     = "/content/drive/MyDrive/cm_hybrid_rds.png"
CURVES_SAVE_PATH = "/content/drive/MyDrive/curves_hybrid_rds.png"

# ==============================================================================
# DATA GENERATORS
# ==============================================================================
# Augmentation pipeline matches the oversampling pipeline (Albumentations was
# used offline; this Keras pipeline mimics it during training for consistency).

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,               # ±20°
    zoom_range=0.10,                 # ±10%
    width_shift_range=0.08,          # ±8%
    height_shift_range=0.08,         # ±8%
    horizontal_flip=True,            # p = 0.5
    brightness_range=[0.88, 1.12],   # ±12%
    fill_mode='reflect'
)

val_datagen  = ImageDataGenerator(rescale=1.0 / 255.0)
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

class_names = list(train_generator.class_indices.keys())
NUM_CLASSES = len(class_names)

print("=" * 78)
print("DATASET SUMMARY")
print("=" * 78)
print(f"Classes ({NUM_CLASSES}):")
for i, name in enumerate(class_names, 1):
    print(f"  {i:2d}. {name}")
print(f"\nTrain images:       {train_generator.samples}")
print(f"Validation images:  {val_generator.samples}")
print(f"Test images:        {test_generator.samples}")
print("=" * 78)

# ==============================================================================
# MODEL BLOCKS
# ==============================================================================

def residual_block(x, filters):
    """
    Residual Block (He et al., 2016 — ResNet style).

    Structure:
        shortcut: 1×1 conv (channel matching)
        main:     3×3 conv → BN → LeakyReLU → 3×3 conv → BN
        output:   Add([main, shortcut]) → LeakyReLU
    """
    # Shortcut path
    shortcut = Conv2D(filters, kernel_size=1, padding="same")(x)

    # Main path
    x = Conv2D(filters, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.1)(x)

    x = Conv2D(filters, kernel_size=3, padding="same")(x)
    x = BatchNormalization()(x)

    # Merge
    x = Add()([x, shortcut])
    x = LeakyReLU(alpha=0.1)(x)

    return x


def dense_block(x, growth_rate=32):
    """
    Dense Block (Huang et al., 2017 — DenseNet style).

    Structure:
        Two 3×3 convolutions, each concatenated with the running feature map.
        Growth rate g = 32 channels per convolution.
    """
    # Layer 1
    y = Conv2D(growth_rate, kernel_size=3, padding="same", activation="relu")(x)
    x = Concatenate()([x, y])

    # Layer 2
    y = Conv2D(growth_rate, kernel_size=3, padding="same", activation="relu")(x)
    x = Concatenate()([x, y])

    return x


def se_block(x, reduction=4):
    """
    Squeeze-and-Excitation Block (Hu et al., 2018 — channel attention).

    Structure:
        Squeeze:    Global Average Pooling → (C,)
        Excitation: Dense(C/r) → ReLU → Dense(C) → Sigmoid
        Scale:      x ⊙ attention
    """
    filters = x.shape[-1]

    # Squeeze
    se = GlobalAveragePooling2D()(x)

    # Excitation
    se = Dense(filters // reduction, activation="relu")(se)
    se = Dense(filters, activation="sigmoid")(se)

    # Reshape to (1, 1, C) for broadcasting
    se = Reshape((1, 1, filters))(se)

    # Scale
    x = Multiply()([x, se])

    return x


# ==============================================================================
# BUILD HYBRID MODEL
# ==============================================================================

def build_hybrid_model(input_shape=(IMG_SIZE, IMG_SIZE, IMG_CHANNELS),
                        num_classes=NUM_CLASSES):
    """Build the Hybrid (Residual + Dense + SE) CNN."""

    inputs = Input(shape=input_shape, name="input_image")

    # ---- Stem ----
    x = Conv2D(64, kernel_size=3, padding="same", name="stem_conv")(inputs)
    x = BatchNormalization(name="stem_bn")(x)
    x = LeakyReLU(alpha=0.1, name="stem_activation")(x)

    # ---- Block 1: Residual (64) ----
    x = residual_block(x, 64)
    x = MaxPooling2D(pool_size=2, name="block1_pool")(x)

    # ---- Block 2: Dense + SE ----
    x = dense_block(x, growth_rate=32)
    x = se_block(x, reduction=4)
    x = MaxPooling2D(pool_size=2, name="block2_pool")(x)

    # ---- Block 3: Residual (128) ----
    x = residual_block(x, 128)
    x = MaxPooling2D(pool_size=2, name="block3_pool")(x)

    # ---- Block 4: Dense + SE ----
    x = dense_block(x, growth_rate=32)
    x = se_block(x, reduction=4)
    x = MaxPooling2D(pool_size=2, name="block4_pool")(x)

    # ---- Block 5: Residual (256), no pooling ----
    x = residual_block(x, 256)

    # ---- Classification Head ----
    x = GlobalAveragePooling2D(name="gap")(x)

    x = Dense(512, name="fc1")(x)
    x = BatchNormalization(name="fc1_bn")(x)
    x = LeakyReLU(alpha=0.1, name="fc1_activation")(x)
    x = Dropout(0.5, name="fc1_dropout")(x)

    x = Dense(256, name="fc2")(x)
    x = LeakyReLU(alpha=0.1, name="fc2_activation")(x)
    x = Dropout(0.3, name="fc2_dropout")(x)

    outputs = Dense(num_classes, activation="softmax", name="predictions")(x)

    model = Model(inputs=inputs, outputs=outputs, name="Hybrid_RDS")

    return model


# ---- Instantiate ----
model = build_hybrid_model()

# ==============================================================================
# COMPILE MODEL
# ==============================================================================
optimizer = AdamW(learning_rate=1e-4, weight_decay=1e-4)

model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

print("\n" + "=" * 78)
print("MODEL SUMMARY")
print("=" * 78)
model.summary()

# ==============================================================================
# CLASS WEIGHTS (balanced)
# ==============================================================================
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights_array))

print("\n" + "=" * 78)
print("CLASS WEIGHTS")
print("=" * 78)
for i, name in enumerate(class_names):
    print(f"  {name:30s}: {class_weights[i]:.4f}")

# ==============================================================================
# CALLBACKS
# ==============================================================================
callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

# ==============================================================================
# TRAIN MODEL
# ==============================================================================
print("\n" + "=" * 78)
print("STARTING TRAINING")
print("=" * 78)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print("\n" + "=" * 78)
print("TRAINING COMPLETE")
print("=" * 78)

# ==============================================================================
# SAVE TRAINING HISTORY
# ==============================================================================
history_dict = {
    "accuracy":     [float(v) for v in history.history["accuracy"]],
    "val_accuracy": [float(v) for v in history.history["val_accuracy"]],
    "loss":         [float(v) for v in history.history["loss"]],
    "val_loss":     [float(v) for v in history.history["val_loss"]],
}
with open(HISTORY_SAVE_PATH, "w") as f:
    json.dump(history_dict, f, indent=2)

print(f"\n✅ Model saved to:   {MODEL_SAVE_PATH}")
print(f"✅ History saved to: {HISTORY_SAVE_PATH}")

# ==============================================================================
# EVALUATE ON TEST SET
# ==============================================================================
print("\n" + "=" * 78)
print("TEST SET EVALUATION")
print("=" * 78)

test_pred_probs = model.predict(test_generator, verbose=1)
test_pred = np.argmax(test_pred_probs, axis=1)
test_true = test_generator.classes

test_acc = accuracy_score(test_true, test_pred) * 100

print(f"\nTest Accuracy: {test_acc:.2f}%\n")
print("Classification Report:")
print(classification_report(test_true, test_pred, target_names=class_names, digits=4))

# ---- Confusion Matrix ----
cm = confusion_matrix(test_true, test_pred)

plt.figure(figsize=(13, 11))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
    cbar_kws={"label": "Number of Samples"}
)
plt.title("Confusion Matrix — Hybrid (Residual + Dense + SE) on Test Set",
          fontsize=13, pad=12)
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("True Label", fontsize=11)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(CM_SAVE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"\n✅ Confusion matrix saved to: {CM_SAVE_PATH}")

# ==============================================================================
# PLOT TRAINING CURVES
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history["accuracy"],     label="Train",      linewidth=2)
axes[0].plot(history.history["val_accuracy"], label="Validation", linewidth=2)
axes[0].set_xlabel("Epoch",    fontsize=11)
axes[0].set_ylabel("Accuracy", fontsize=11)
axes[0].set_title("Model Accuracy", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history["loss"],     label="Train",      linewidth=2)
axes[1].plot(history.history["val_loss"], label="Validation", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Loss",  fontsize=11)
axes[1].set_title("Model Loss", fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CURVES_SAVE_PATH, dpi=300, bbox_inches="tight")
plt.show()

print(f"✅ Training curves saved to: {CURVES_SAVE_PATH}")

# ==============================================================================
# SAVE EVALUATION RESULTS (JSON)
# ==============================================================================
results = {
    "test_accuracy": float(test_acc),
    "classification_report": classification_report(
        test_true, test_pred, target_names=class_names, output_dict=True
    ),
    "confusion_matrix": cm.tolist(),
    "class_names": class_names,
}
with open("/content/drive/MyDrive/results_hybrid_rds.json", "w") as f:
    json.dump(results, f, indent=2)

print("✅ Results JSON saved.")

# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
print("\n" + "=" * 78)
print("FINAL SUMMARY — Hybrid (Residual + Dense + SE)")
print("=" * 78)
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Total Test Samples: {len(test_true)}")
print("=" * 78)

PAPER MEIN YEH KAISE LIKHNA HAI
"The proposed hybrid architecture combines three established design principles:

(i) Residual connections (He et al.) with 2 convolutional layers per block, using 1×1 shortcut convolutions for channel matching;

(ii) Dense connectivity (Huang et al.) with a growth rate of 32 and two 3×3 convolutions per dense layer, followed by channel-wise concatenation;

(iii) Squeeze-and-Excitation attention (Hu et al.) with a reduction ratio of 4, applying ReLU in the squeeze layer and sigmoid for gating.

All convolutional layers use 3×3 kernels, and LeakyReLU (α=0.1) is used throughout, which is standard for residual networks. The classification head consists of two fully connected layers (512, 256) with batch normalization and dropout (0.5, 0.3). This configuration was chosen as it provides a strong balance between expressiveness, parameter count, and training stability."

In [ ]:
# ==============================================================================
# GRAD-CAM VISUALIZATION
# ==============================================================================
# Yeh code aap ke TRAINED model par Grad-CAM apply karta hai.
# Colab mein training ke BAAD chalayein.
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# ------------------------------------------------------------------------------
# Load trained model
# ------------------------------------------------------------------------------
MODEL_PATH = "/content/drive/MyDrive/rice_leaf_hybrid/hybrid_rds_best.keras"
model = load_model(MODEL_PATH)

# ------------------------------------------------------------------------------
# Grad-CAM function
# ------------------------------------------------------------------------------
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generate Grad-CAM heatmap for a given image.
    
    Args:
        img_array: Preprocessed image array (1, 224, 224, 3)
        model: Trained Keras model
        last_conv_layer_name: Name of last conv layer (for this model: 'block5_pool' or similar)
        pred_index: Optional class index for which to generate CAM
    
    Returns:
        heatmap: Normalized heatmap (224, 224)
    """
    # Create a model that maps input -> (last_conv_output, final_prediction)
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    
    # Compute gradients
    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    
    # Gradients of predicted class wrt last conv output
    grads = tape.gradient(class_channel, last_conv_output)
    
    # Global average pooling of gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    # Weight the conv output by pooled gradients
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    
    # Normalize
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    
    return heatmap.numpy()


def display_gradcam(img_path, model, last_conv_layer_name, class_names, alpha=0.4):
    """
    Display Grad-CAM heatmap overlaid on original image.
    """
    # Load image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    preds = model.predict(img_array, verbose=0)
    pred_class = np.argmax(preds[0])
    confidence = np.max(preds[0]) * 100
    
    # Generate heatmap
    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
    
    # Resize heatmap to match image size
    heatmap = cv2.resize(heatmap, (224, 224))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    # Superimpose
    original = np.uint8(255 * img_array[0])
    superimposed = cv2.addWeighted(original, 1 - alpha, heatmap, alpha, 0)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(original)
    axes[0].set_title("Original Image")
    axes[0].axis("off")
    
    axes[1].imshow(heatmap)
    axes[1].set_title("Grad-CAM Heatmap")
    axes[1].axis("off")
    
    axes[2].imshow(superimposed)
    axes[2].set_title(f"Overlay\nPredicted: {class_names[pred_class]}\nConfidence: {confidence:.2f}%")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    return pred_class, confidence


# ------------------------------------------------------------------------------
# Find the last conv layer name
# ------------------------------------------------------------------------------
print("=" * 78)
print("MODEL LAYERS (LAST 10)")
print("=" * 78)
for i, layer in enumerate(model.layers[-10:], start=len(model.layers) - 10):
    print(f"  {i:3d}. {layer.name:30s} - {layer.__class__.__name__}")

# For this model, the last conv-related layer is typically 'block5_pool' or 'residual'
# Aap manually check karein aur yahan likhein:
LAST_CONV_LAYER = "block4_pool"   # ← Change this based on model
# OR run the print above and pick the last Conv2D layer name

# ------------------------------------------------------------------------------
# Generate Grad-CAM for test images
# ------------------------------------------------------------------------------
TEST_DIR = "/content/rice_dataset/rice_leaf_disease_dataset/test"
class_names = sorted(os.listdir(TEST_DIR))

# Pick 4 sample images (one per random class)
import random
random.seed(42)

sample_paths = []
for cls in random.sample(class_names, 4):
    cls_path = os.path.join(TEST_DIR, cls)
    img_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if img_files:
        sample_paths.append(os.path.join(cls_path, random.choice(img_files)))

# Display Grad-CAM for each
for img_path in sample_paths:
    print(f"\n📷 Image: {img_path}")
    display_gradcam(img_path, model, LAST_CONV_LAYER, class_names)